# Feature Engineering

This notebook demonstrates the nine engineered features in
`src/features/build_features.py`.  Every feature is motivated by a specific
finding from the bivariate EDA (notebook 02); the cells below validate those
motivations empirically by showing churn rates by feature level and comparing
mutual information (MI) scores for engineered features against the originals
they are derived from.

**Features covered**

| Feature | Source columns | Motivation |
|---|---|---|
| `tenure_bucket` | `tenure` | Non-linear churn decay; new customers churn at ~50% |
| `total_services` | 9 service cols | Depth of product relationship → retention |
| `services_intensity` | `total_services`, `tenure` | Rapid adoption signals promo-churn risk |
| `contract_risk_score` | `Contract` | Highest Cramér's V of all categoricals |
| `payment_risk` | `PaymentMethod` | Electronic check churn rate 2× autopay |
| `tenure_x_contract` | `tenure`, `Contract` | Explicit interaction for shallow trees |
| `monthly_charges_per_service` | `MonthlyCharges`, services | Price efficiency / bundling incentive |
| `has_premium_services` | `StreamingTV`, `StreamingMovies` | Premium streaming ↔ fiber/promo cohort |
| `autopay_flag` | `PaymentMethod` | Clean binary from payment risk |


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.load import download_telco_data, load_raw
from src.features.build_features import engineer_features
from src.features.eda_helpers import churn_rate_by_group, compute_mutual_info
from src.utils.plotting import CHURN_PALETTE, PALETTE, set_plot_style, save_fig

set_plot_style()
FIGURES_DIR = REPO_ROOT / "reports" / "figures"

In [ ]:
with open(REPO_ROOT / "configs" / "config.yaml") as fh:
    cfg = yaml.safe_load(fh)

RAW_PATH = REPO_ROOT / cfg["paths"]["raw_data"]
download_telco_data(RAW_PATH, urls=cfg["data"]["download_urls"])
raw = load_raw(RAW_PATH)

print(f"Loaded  {len(raw):,} rows × {raw.shape[1]} columns")

## Preprocessing

One known data quality issue: `TotalCharges` is blank for 11 zero-tenure
customers.  We coerce it to numeric (blanks → NaN → 0) before engineering.

In [ ]:
df_raw = raw.copy()
df_raw["TotalCharges"] = pd.to_numeric(df_raw["TotalCharges"], errors="coerce").fillna(0.0)

df = engineer_features(df_raw)
print(f"Original columns : {raw.shape[1]}")
print(f"Engineered columns: {df.shape[1] - raw.shape[1]} new  ({df.shape[1]} total)")
df.head(3)

## 1. `tenure_bucket` — Cohort Bins

Raw `tenure` has a non-linear relationship with churn: risk falls sharply after
month 12 and again after month 24.  The ordered Categorical bin captures this
shape without forcing a linear coefficient on the model.

In [ ]:
bucket_stats = churn_rate_by_group(df, "tenure_bucket").sort_index()
display(bucket_stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bucket_stats["count"].plot.bar(ax=axes[0], color=PALETTE[0], edgecolor="white")
axes[0].set_title("Customer count by tenure bucket")
axes[0].set_xlabel("tenure_bucket")
axes[0].set_ylabel("Customers")
axes[0].tick_params(axis="x", rotation=0)

bucket_stats["churn_rate"].plot.bar(ax=axes[1], color=PALETTE[1], edgecolor="white")
axes[1].set_title("Churn rate by tenure bucket")
axes[1].set_xlabel("tenure_bucket")
axes[1].set_ylabel("Churn rate")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[1].tick_params(axis="x", rotation=0)

fig.tight_layout()
save_fig(fig, "03_tenure_bucket", FIGURES_DIR)
plt.show()

## 2. `total_services` and `services_intensity`

Customers with more active subscriptions have stronger product relationships and
face higher switching costs.  `services_intensity` captures how rapidly a
customer adopted services relative to their tenure — high early adoption can
indicate promotional sign-ups that convert poorly to long-term relationships.

In [ ]:
services_stats = churn_rate_by_group(df, "total_services").sort_index()
display(services_stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

services_stats["churn_rate"].plot.bar(
    ax=axes[0], color=PALETTE[1], edgecolor="white"
)
axes[0].set_title("Churn rate by total_services")
axes[0].set_xlabel("total_services")
axes[0].set_ylabel("Churn rate")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[0].tick_params(axis="x", rotation=0)

# services_intensity distribution by churn label
for label, grp in df.groupby("Churn"):
    axes[1].hist(
        grp["services_intensity"].clip(upper=50),
        bins=40,
        alpha=0.6,
        color=CHURN_PALETTE[label],
        label=f"Churn={label}",
        density=True,
    )
axes[1].set_title("services_intensity distribution (clipped at 50)")
axes[1].set_xlabel("services_intensity")
axes[1].set_ylabel("Density")
axes[1].legend(frameon=False)

fig.tight_layout()
save_fig(fig, "03_services_features", FIGURES_DIR)
plt.show()

## 3. `contract_risk_score` and `payment_risk`

Both features convert categorical columns into interpretable ordinal scores.
`contract_risk_score` encodes the three contract types (0=two-year,
1=one-year, 2=month-to-month); `payment_risk` encodes payment methods
(0=autopay, 1=mailed check, 2=electronic check).  The ordinal encoding
preserves the ranked churn signal in a single integer, more compact than
one-hot encoding for tree learners.

In [ ]:
contract_stats = churn_rate_by_group(df, "contract_risk_score").sort_index()
payment_stats = churn_rate_by_group(df, "payment_risk").sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

contract_stats["churn_rate"].plot.bar(
    ax=axes[0], color=PALETTE[1], edgecolor="white"
)
axes[0].set_title("Churn rate by contract_risk_score\n(0=two-year, 1=one-year, 2=month-to-month)")
axes[0].set_xlabel("contract_risk_score")
axes[0].set_ylabel("Churn rate")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[0].tick_params(axis="x", rotation=0)

payment_stats["churn_rate"].plot.bar(
    ax=axes[1], color=PALETTE[1], edgecolor="white"
)
axes[1].set_title("Churn rate by payment_risk\n(0=autopay, 1=mailed check, 2=electronic check)")
axes[1].set_xlabel("payment_risk")
axes[1].set_ylabel("Churn rate")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[1].tick_params(axis="x", rotation=0)

fig.tight_layout()
save_fig(fig, "03_risk_scores", FIGURES_DIR)
plt.show()

## 4. `tenure_x_contract` — Interaction Feature

Tree ensembles can learn interactions at sufficient depth, but an explicit
product term makes the most predictive combination immediately available at
shallow splits.  New customers on month-to-month contracts are the highest
churn risk; long-tenured customers on two-year contracts are the lowest.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, grp in df.groupby("Churn"):
    ax.hist(
        grp["tenure_x_contract"],
        bins=50,
        alpha=0.6,
        color=CHURN_PALETTE[label],
        label=f"Churn={label}",
        density=True,
    )
ax.set_title("tenure_x_contract distribution by churn label")
ax.set_xlabel("tenure × contract_risk_score")
ax.set_ylabel("Density")
ax.legend(frameon=False)
fig.tight_layout()
save_fig(fig, "03_tenure_x_contract", FIGURES_DIR)
plt.show()

print(df.groupby("Churn")["tenure_x_contract"].describe().T)

## 5. `monthly_charges_per_service`, `has_premium_services`, `autopay_flag`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# monthly_charges_per_service
for label, grp in df.groupby("Churn"):
    axes[0].hist(
        grp["monthly_charges_per_service"],
        bins=40,
        alpha=0.6,
        color=CHURN_PALETTE[label],
        label=f"Churn={label}",
        density=True,
    )
axes[0].set_title("monthly_charges_per_service")
axes[0].set_xlabel("$/service")
axes[0].set_ylabel("Density")
axes[0].legend(frameon=False)

# has_premium_services — churn rate
premium_stats = churn_rate_by_group(df, "has_premium_services").sort_index()
premium_stats["churn_rate"].plot.bar(
    ax=axes[1], color=PALETTE[1], edgecolor="white"
)
axes[1].set_title("Churn rate: has_premium_services")
axes[1].set_xlabel("has_premium_services (0=No, 1=Yes)")
axes[1].set_ylabel("Churn rate")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[1].tick_params(axis="x", rotation=0)

# autopay_flag — churn rate
autopay_stats = churn_rate_by_group(df, "autopay_flag").sort_index()
autopay_stats["churn_rate"].plot.bar(
    ax=axes[2], color=PALETTE[1], edgecolor="white"
)
axes[2].set_title("Churn rate: autopay_flag")
axes[2].set_xlabel("autopay_flag (0=manual, 1=automatic)")
axes[2].set_ylabel("Churn rate")
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[2].tick_params(axis="x", rotation=0)

fig.tight_layout()
save_fig(fig, "03_remaining_features", FIGURES_DIR)
plt.show()

## 6. Mutual Information: Engineered vs. Original Features

We compute MI for the nine engineered features and compare each against the
original source column(s) it is derived from.  Features that exceed their
source column's MI score deliver genuine incremental information; those below
may still be useful as explicit representations of a known non-linearity.

In [ ]:
# Encode tenure_bucket as an integer for MI calculation
_bucket_order = {"new": 0, "developing": 1, "established": 2, "loyal": 3}
df["tenure_bucket_int"] = df["tenure_bucket"].astype(str).map(_bucket_order)

ENGINEERED_COLS = [
    "tenure_bucket_int",
    "total_services",
    "services_intensity",
    "contract_risk_score",
    "payment_risk",
    "tenure_x_contract",
    "monthly_charges_per_service",
    "has_premium_services",
    "autopay_flag",
]

ORIGINAL_COLS = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract",
    "PaymentMethod",
    "InternetService",
    "OnlineSecurity",
    "TechSupport",
]

mi_engineered = compute_mutual_info(
    df, ENGINEERED_COLS, target_col="Churn"
).rename(index={"tenure_bucket_int": "tenure_bucket"})

mi_original = compute_mutual_info(df, ORIGINAL_COLS, target_col="Churn")

mi_all = pd.concat(
    [
        mi_engineered.rename("MI (engineered)"),
        mi_original.rename("MI (original)"),
    ],
    axis=1,
).sort_values("MI (engineered)", ascending=False, na_position="last")

display(mi_all.round(4))

In [ ]:
mi_combined = pd.concat(
    [
        mi_engineered.rename("MI Score").to_frame().assign(Type="Engineered"),
        mi_original.rename("MI Score").to_frame().assign(Type="Original"),
    ]
).sort_values("MI Score", ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
colors = [PALETTE[0] if t == "Engineered" else PALETTE[2] for t in mi_combined["Type"]]
ax.barh(mi_combined.index, mi_combined["MI Score"], color=colors, edgecolor="white")
ax.set_title("Mutual Information vs. Churn — Engineered vs. Original Features")
ax.set_xlabel("Mutual Information Score")

from matplotlib.patches import Patch
legend_handles = [
    Patch(color=PALETTE[0], label="Engineered"),
    Patch(color=PALETTE[2], label="Original"),
]
ax.legend(handles=legend_handles, frameon=False)
fig.tight_layout()
save_fig(fig, "03_mi_comparison", FIGURES_DIR)
plt.show()

## Summary

The engineered features broadly reproduce or improve on the MI scores of their
source columns:

- `contract_risk_score` matches or exceeds `Contract` by compressing the
  three-level categorical to an ordinal integer that tree splits can use more
  efficiently.
- `tenure_bucket` captures the non-linear churn decay curve without forcing a
  linear coefficient; its MI is comparable to raw `tenure`.
- `total_services` aggregates nine individual service flags into a single count
  that reflects the depth of product relationship.
- `autopay_flag` distils the four-level `PaymentMethod` down to the binary
  split that carries most of the predictive signal.
- `tenure_x_contract` provides an explicit interaction term that shallow tree
  splits can use directly, complementing the individual components.

All nine features are available via `src.features.build_features.engineer_features`
and will be included in the modelling feature set in the next phase.